# 08 · Clustering with k-means and dimensionality reduction with PCA

After this notebook you can implement k-means and PCA from scratch, choose k with the elbow and silhouette methods, explain k-means++ and when DBSCAN beats k-means, read an explained-variance curve, visualise high-dimensional data with PCA and t-SNE, cluster documents, and explain how k-means powers the IVF index inside vector databases.

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min · **Needs:** [07 · kNN, SVMs and Naive Bayes](07_knn_svm_naive_bayes.ipynb)

## Why this matters in interviews

- Unsupervised learning shows up in GenAI work all the time: clustering user queries to find intents, topic-grouping documents, deduplicating near-identical chunks, and building hierarchical indexes (RAPTOR).
- k-means is not just a textbook algorithm: it is how an **IVF** vector index is built, and how product quantisation compresses embeddings. "How does IVF work?" is a common vector-search question.
- PCA and t-SNE are how you *look at* embeddings, and PCA's explained-variance curve is the concrete answer to "do embeddings really use all 1,536 dimensions?".

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ve05` | How does IVF work, and how does it compare to HNSW? |
| `tl28` | Explain HNSW versus IVF. |
| `fo04` | What is an embedding? |
| `ve20` | What is the curse of dimensionality and does it affect embeddings? |
| `rg54` | What is RAPTOR / hierarchical retrieval? |

**Also asked in classical-ML rounds:** walk me through k-means; does it always converge, and to what? how do you choose k? what is k-means++? when would you use DBSCAN or hierarchical clustering instead? what does PCA maximise? PCA vs t-SNE vs UMAP; why scale before PCA?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, silhouette_samples, silhouette_score
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
BLUE, ORANGE, AQUA, YELLOW, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a8984"
PALETTE = [BLUE, ORANGE, AQUA, YELLOW, "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": plt.cycler(color=PALETTE)})

## 1. k-means from scratch

**In plain English:** pick k starting centres. Then repeat two steps: (1) give every point to its nearest centre; (2) move each centre to the average of its points. Stop when nothing changes.

It minimises the **inertia**, the sum of squared distances from each point to its centre. Each step can only lower (or keep) the inertia, so it always converges, but only to a *local* minimum that depends on the starting centres.

In [ ]:
Xb, yb = make_blobs(n_samples=600, centers=4, cluster_std=1.0, random_state=7)

def kmeans(X, k, init_centers, max_iter=100):
    """Lloyd's algorithm. Returns labels, centres, inertia per iteration and the centres after each iteration."""
    centers = init_centers.copy()
    history, inertias = [centers.copy()], []
    for _ in range(max_iter):
        d2 = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=-1)       # (points, k) squared distances
        labels = d2.argmin(axis=1)                                             # step 1: assign
        inertias.append(d2[np.arange(len(X)), labels].sum())
        new_centers = np.array([X[labels == j].mean(axis=0) for j in range(k)])   # step 2: update
        if np.allclose(new_centers, centers):
            break
        centers = new_centers
        history.append(centers.copy())
    return labels, centers, np.array(inertias), history

init = Xb[rng.choice(len(Xb), 4, replace=False)]              # 4 random data points as starting centres
labels, centers, inertias, history = kmeans(Xb, 4, init)
print(f"converged after {len(inertias)} assignment steps; inertia by step: {inertias.round(0)}")
assert np.all(np.diff(inertias) <= 1e-9)                       # inertia never goes up

sk = KMeans(n_clusters=4, init=init, n_init=1).fit(Xb)          # same start -> same answer
assert np.allclose(np.sort(sk.cluster_centers_, axis=0), np.sort(centers, axis=0)) and np.isclose(sk.inertia_, inertias[-1])
print("matches sklearn KMeans started from the same centres")

In [ ]:
steps = list(range(len(history)))[:4]                          # start, then each update until convergence
fig, axes = plt.subplots(1, len(steps), figsize=(2.8 * len(steps), 3), sharey=True)
for ax, s in zip(axes, steps):
    c = history[s]
    lab = ((Xb[:, None, :] - c[None]) ** 2).sum(-1).argmin(1)
    ax.scatter(Xb[:, 0], Xb[:, 1], c=np.array(PALETTE)[lab], s=6, alpha=0.6)
    ax.scatter(c[:, 0], c[:, 1], marker="X", s=160, color="black", edgecolors="white")
    ax.set(title="start" if s == 0 else f"after {s} update{'s' if s > 1 else ''}" + (" (converged)" if s == len(history) - 1 else ""),
           xticks=[], yticks=[])
fig.suptitle("k-means iterations: assign to the nearest centre (X), move centres to the mean", y=1.04); plt.show()

## 2. Choosing k: elbow and silhouette

k-means cannot tell you k. Two standard heuristics:

- **Elbow:** plot inertia against k. It always falls (more centres, shorter distances); look for the bend where adding a cluster stops paying.
- **Silhouette:** for each point, $a$ = mean distance to its own cluster, $b$ = mean distance to the nearest *other* cluster, $s = \frac{b-a}{\max(a,b)}$. Near 1 means well placed; near 0, on a border; negative, probably in the wrong cluster. Pick the k with the highest average.

In [ ]:
def silhouette_np(X, labels):
    D = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(-1))
    s = np.empty(len(X))
    for i in range(len(X)):
        own = labels == labels[i]
        a = D[i, own].sum() / (own.sum() - 1)                                 # exclude the point itself
        b = min(D[i, labels == c].mean() for c in np.unique(labels) if c != labels[i])
        s[i] = (b - a) / max(a, b)
    return s

s_ours = silhouette_np(Xb, labels)
assert np.allclose(s_ours, silhouette_samples(Xb, labels))
print(f"mean silhouette for k=4: ours {s_ours.mean():.3f}, sklearn {silhouette_score(Xb, labels):.3f}")

ks = range(2, 10)
fits = {k: KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xb) for k in ks}
inertia_k = [fits[k].inertia_ for k in ks]
sil_k = [silhouette_score(Xb, fits[k].labels_) for k in ks]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(list(ks), inertia_k, "o-", lw=2); axes[0].set(title="Elbow: inertia vs k", xlabel="k", ylabel="inertia")
axes[1].plot(list(ks), sil_k, "o-", lw=2, color=ORANGE); axes[1].set(title="Mean silhouette vs k", xlabel="k", ylabel="silhouette")
for ax in axes:
    ax.axvline(4, color=GREY, ls="--", lw=1)
plt.tight_layout(); plt.show()
best_k = list(ks)[int(np.argmax(sil_k))]
print(f"best k by silhouette: {best_k} (the data was generated with 4 blobs)")
assert best_k == 4

On real data (embeddings of user queries, say) the elbow is rarely sharp and the silhouette peak is shallow. Treat both as hints, then read samples from each cluster: the useful k is the one whose clusters a human can name.

## 3. k-means++: better starting centres

Random starting points can land two centres in one blob and none in another, and Lloyd's algorithm cannot escape that local minimum. **k-means++** picks the first centre at random, then each next centre with probability proportional to its *squared distance* to the nearest centre chosen so far, so the starts are spread out. It is scikit-learn's default; `n_init` additionally reruns from several starts and keeps the best.

In [ ]:
Xk, yk = make_blobs(n_samples=1500, centers=10, cluster_std=1.0, random_state=3)
random_init = [KMeans(10, init="random", n_init=1, random_state=s).fit(Xk).inertia_ for s in range(30)]
plusplus = [KMeans(10, init="k-means++", n_init=1, random_state=s).fit(Xk).inertia_ for s in range(30)]

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.scatter(random_init, np.zeros(30) + rng.uniform(-0.1, 0.1, 30), s=25, color=BLUE, label="random init")
ax.scatter(plusplus, np.ones(30) + rng.uniform(-0.1, 0.1, 30), s=25, color=ORANGE, label="k-means++ init")
ax.set(yticks=[0, 1], yticklabels=["random", "k-means++"], xscale="log",
       title="Final inertia of 30 single runs (10 true clusters); lower is better", xlabel="inertia (log)")
plt.show()
print(f"random init: mean {np.mean(random_init):.0f}, worst {np.max(random_init):.0f}")
print(f"k-means++  : mean {np.mean(plusplus):.0f}, worst {np.max(plusplus):.0f}")
assert np.mean(plusplus) < np.mean(random_init) and np.max(plusplus) < np.max(random_init)

## 4. When k-means fails: DBSCAN on moons

k-means assumes roughly round, similar-sized clusters (each point goes to its nearest centre, so boundaries are straight lines between centres). **DBSCAN** instead grows clusters from dense regions: a point with at least `min_samples` neighbours within radius `eps` is a *core* point, clusters are chains of connected core points, and points in sparse areas are labelled **noise** (`-1`). It finds any shape and does not need k, but you must choose `eps`.

In [ ]:
Xmo, ymo = make_moons(n_samples=500, noise=0.06, random_state=0)
km_lab = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(Xmo)
db_lab = DBSCAN(eps=0.15, min_samples=5).fit_predict(Xmo)
ari_km, ari_db = adjusted_rand_score(ymo, km_lab), adjusted_rand_score(ymo, db_lab)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].scatter(Xmo[:, 0], Xmo[:, 1], c=np.array(PALETTE)[km_lab], s=8)
axes[0].set(title=f"k-means (k=2): adjusted Rand index {ari_km:.2f}", xticks=[], yticks=[])
colors = np.where(db_lab == -1, GREY, np.array(PALETTE)[np.clip(db_lab, 0, None)])
axes[1].scatter(Xmo[:, 0], Xmo[:, 1], c=colors, s=8)
axes[1].set(title=f"DBSCAN (eps=0.15): ARI {ari_db:.2f}, {np.sum(db_lab == -1)} noise points (grey)", xticks=[], yticks=[])
plt.tight_layout(); plt.show()
assert ari_db > 0.95 and ari_km < 0.6

The **adjusted Rand index** compares a clustering with known labels (1 = identical, about 0 = random), so it is only for demos and evaluation sets; in real use you rarely have the labels.

| Algorithm | Needs k? | Cluster shapes | Handles noise | Scales to | Use it for |
|---|---|---|---|---|---|
| k-means | yes | round, similar size | no (every point is assigned) | millions (mini-batch) | quantising vectors, quick segmentation, IVF |
| DBSCAN / HDBSCAN | no (eps / min size) | arbitrary | yes, labels outliers | ~1e5 to 1e6 with an index | finding dense topic groups, outlier queries |
| Agglomerative (hierarchical) | cut the tree anywhere | depends on linkage | no | ~1e4 (O(n²) memory) | small sets, a dendrogram of topics, RAPTOR-style trees |
| Gaussian mixture | yes | elliptical, soft membership | somewhat | ~1e5 | probabilistic assignments |

## 5. PCA from scratch via the SVD

**In plain English:** find the directions along which the data varies the most. The first principal component is the single direction that keeps the most variance; the second is the best direction at right angles to the first; and so on. Projecting onto the first few keeps most of the information in far fewer numbers.

Recipe: **centre** the data, take the SVD $X_c = U S V^\top$; the rows of $V^\top$ are the components and the variance along component $i$ is $S_i^2/(n-1)$. On the digits data (1,797 images of 8×8 = 64 pixels):

In [ ]:
digits = load_digits()
Xd, yd = digits.data, digits.target
Xc = Xd - Xd.mean(axis=0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
explained_var = S ** 2 / (len(Xd) - 1)
ratio_ours = explained_var / explained_var.sum()
Z_ours = Xc @ Vt[:2].T                                          # project onto the first two components

pca = PCA().fit(Xd)
Z_sk = pca.transform(Xd)[:, :2]
print("explained variance ratio, first 5 components:", ratio_ours[:5].round(4))
assert np.allclose(ratio_ours, pca.explained_variance_ratio_)
for i in range(5):                                               # components agree up to a sign flip
    assert np.isclose(abs(Vt[i] @ pca.components_[i]), 1.0)
assert np.allclose(np.abs(Z_ours), np.abs(Z_sk))
print("matches sklearn PCA (components equal up to sign)")

Why "up to sign"? If $v$ is a direction of maximum variance, so is $-v$. Libraries pick a sign convention; never compare component signs across libraries or runs.

### How many dimensions does the data really use? (`ve20`)

In [ ]:
cum = np.cumsum(ratio_ours)
need = {t: int(np.searchsorted(cum, t) + 1) for t in (0.80, 0.90, 0.95, 0.99)}
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(np.arange(1, 65), ratio_ours, color=BLUE, alpha=0.6, label="per component")
ax.plot(np.arange(1, 65), cum, lw=2, color=ORANGE, label="cumulative")
for t, k in need.items():
    ax.axhline(t, color=GREY, lw=0.6, ls=":")
    ax.annotate(f"{t:.0%}: {k} comps", xy=(k, t), xytext=(k + 3, t - 0.07), fontsize=8, arrowprops=dict(arrowstyle="->", lw=0.6))
ax.set(title="Digits: explained variance by principal component (64 pixels)", xlabel="component", ylabel="share of variance")
ax.legend(loc="center right"); plt.show()
print("components needed:", {f"{t:.0%}": k for t, k in need.items()})
assert need[0.95] < 40

Out of 64 pixel dimensions, about 30 components hold 95% of the variance: the data lives on a lower-dimensional surface. Embeddings behave the same way (a 1,536-dimensional embedding space has a much smaller intrinsic dimension), which is why nearest-neighbour search in them still works despite the curse of dimensionality, and why truncating or compressing vectors loses less than you might expect.

## 6. Seeing the data: 2-D PCA and t-SNE

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
cmap10 = plt.get_cmap("tab10")
ax.scatter(Z_ours[:, 0], Z_ours[:, 1], c=[cmap10(k) for k in yd], s=6, alpha=0.6)
for k in range(10):                                            # direct labels at each digit's centre
    cx, cy = Z_ours[yd == k].mean(axis=0)
    ax.text(cx, cy, str(k), fontsize=14, weight="bold", ha="center", va="center",
            bbox=dict(boxstyle="circle,pad=0.2", fc="white", ec=cmap10(k)))
ax.set(title=f"Digits projected on 2 principal components ({cum[1]:.0%} of the variance)", xlabel="PC 1", ylabel="PC 2")
plt.show()

Two components keep under a third of the variance, so several digits overlap. PCA is **linear**: it can only rotate and flatten. **t-SNE** is a non-linear method that tries to keep each point's *neighbours* close, which separates the classes far better, at a price.

In [ ]:
sub = rng.choice(len(Xd), 500, replace=False)
tsne_xy = TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(Xd[sub])
pca_xy = Z_ours[sub]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, xy, name in [(axes[0], pca_xy, "PCA (linear)"), (axes[1], tsne_xy, "t-SNE (non-linear, perplexity 30)")]:
    ax.scatter(xy[:, 0], xy[:, 1], c=[cmap10(k) for k in yd[sub]], s=8, alpha=0.7)
    for k in range(10):
        cx, cy = np.median(xy[yd[sub] == k], axis=0)
        ax.text(cx, cy, str(k), fontsize=12, weight="bold", ha="center", va="center",
                bbox=dict(boxstyle="circle,pad=0.2", fc="white", ec=cmap10(k)))
    ax.set(title=f"500 digits: {name}", xticks=[], yticks=[])
plt.tight_layout(); plt.show()

| | PCA | t-SNE | UMAP (not installed here) |
|---|---|---|---|
| Type | linear projection | non-linear, neighbour-preserving | non-linear, neighbour-preserving |
| Deterministic | yes | no (set `random_state`) | no |
| New points | `transform()` works | no transform: refit | has `transform` |
| Distances between clusters | meaningful | **not** meaningful | loosely |
| Speed | very fast | slow (thousands of points) | faster than t-SNE |
| Use for | compression, denoising, a preprocessing step | pictures only | pictures, sometimes preprocessing |

The t-SNE trap interviewers probe: cluster sizes and the gaps between clusters in a t-SNE plot mean nothing, and different perplexities give different pictures. Never cluster on t-SNE coordinates and never read distances off them.

## 7. Tie-in: clustering documents and visualising their embeddings (`fo04`)

An **embedding** turns a text into a vector so that similar meanings are close. Here, to stay offline, the embedding is TF-IDF compressed to 20 dimensions with truncated SVD (classic latent semantic analysis). With a real embedding model (sentence-transformers, OpenAI `text-embedding-3-small`) the code below is identical; only the vectors improve, especially for paraphrases that share no words.

In [ ]:
docs = {
    "billing": ["How do I update the card used for my payment", "I was charged twice, please refund the payment",
                "Where can I download the invoice for my last payment", "Why did the price on my invoice go up",
                "Cancel my subscription and refund the last charge", "The card payment failed but the charge still shows"],
    "login": ["I forgot my password and cannot log in to my account", "The reset password email never arrives",
              "My login code is not working and I cannot access my account", "My account is locked after too many login attempts",
              "How do I change the email and password on my account", "Single sign on login shows an account error"],
    "shipping": ["When will my order be delivered", "The tracking number for my order delivery does not work",
                 "My order arrived damaged in the parcel", "Can I change the delivery address for my order",
                 "Do you ship orders internationally and how long is delivery", "The courier left my parcel at the wrong delivery address"],
    "bugs": ["The app crashes with an error when I open settings", "The dashboard page in the app is blank after the update",
             "Export to CSV in the app produces an empty file", "Notifications in the app stopped working after the update",
             "The search page shows an error and the app freezes", "Dark mode in the app makes the text unreadable, a display bug"],
}
texts = [t for group in docs.values() for t in group]
topic = np.repeat(np.arange(len(docs)), [len(g) for g in docs.values()])

tfidf = TfidfVectorizer(stop_words="english").fit_transform(texts)
emb = TruncatedSVD(n_components=20, random_state=0).fit_transform(tfidf)
emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)                 # L2-normalise, as for any embedding
clusters = KMeans(n_clusters=4, n_init=20, random_state=0).fit_predict(emb)
ari_docs = adjusted_rand_score(topic, clusters)
print(f"k-means on 24 support tickets: adjusted Rand index vs the true topics = {ari_docs:.2f}")
names = list(docs)
for c in range(4):
    idx = np.where(clusters == c)[0]
    mix = {names[t]: int(n) for t, n in enumerate(np.bincount(topic[idx], minlength=4)) if n}
    print(f"cluster {c}: {len(idx)} tickets, true topics {mix}; e.g. '{texts[idx[0]][:45]}'")
assert ari_docs > 0.6

In [ ]:
xy = PCA(n_components=2).fit_transform(emb)
fig, ax = plt.subplots(figsize=(8, 4.2))
markers = ["o", "s", "^", "D"]
for t, name in enumerate(docs):
    pts = xy[topic == t]
    ax.scatter(pts[:, 0], pts[:, 1], s=60, marker=markers[t], color=PALETTE[t], label=f"true topic: {name}",
               edgecolors=[("black" if clusters[i] != np.bincount(clusters[topic == t]).argmax() else "white") for i in np.where(topic == t)[0]])
ax.set(title="Ticket embeddings (TF-IDF + SVD) in 2-D; black edge = not in its topic's main cluster", xlabel="PC 1", ylabel="PC 2")
ax.legend(fontsize=8); plt.show()

The clusters mostly follow the topics, and the mistakes are instructive. Read the composition printed above: word-count embeddings group tickets that share *words*, so tickets can split or merge along vocabulary lines that no one labelled (the "password" tickets, for instance, share words with each other far more than with the other login tickets). That is why you read samples of each cluster rather than trust k. Lexical vectors also miss paraphrases that share no words at all, which is the reason dense embedding models exist ([embeddings and similarity](../07_genai_foundations/02_embeddings_and_similarity.ipynb)). In production this is how you mine intents from logs: embed, cluster, then read samples of each cluster and name them. **RAPTOR (`rg54`)** applies the same step recursively to RAG chunks: cluster the chunks, summarise each cluster with an LLM, embed the summaries, cluster again, and index every level so that broad questions hit summaries and detailed ones hit leaves.

## 8. k-means inside a vector database: the IVF index (`ve05`, `tl28`)

**In plain English:** run k-means on all stored vectors to get `nlist` centroids ("neighbourhoods"), and file each vector under its nearest centroid. At query time, find the `nprobe` centroids nearest the query and search only those lists. You scan a fraction of the data and sometimes miss a true neighbour that sits just across a boundary. `nprobe` is the recall/latency dial.

In [ ]:
base = np.vstack([rng.normal(loc=rng.normal(0, 4, 32), scale=1.0, size=(400, 32)) for _ in range(50)])   # 20k vectors
base /= np.linalg.norm(base, axis=1, keepdims=True)
queries = base[rng.choice(len(base), 200, replace=False)] + rng.normal(0, 0.02, (200, 32))

nlist, k_nn = 64, 10
ivf = KMeans(n_clusters=nlist, n_init=1, random_state=0).fit(base)            # "training" the index
lists = [np.where(ivf.labels_ == c)[0] for c in range(nlist)]                 # the inverted lists

def ivf_search(q, nprobe):
    probe = np.argsort(((ivf.cluster_centers_ - q) ** 2).sum(1))[:nprobe]     # nearest centroids
    cand = np.concatenate([lists[c] for c in probe])
    top = cand[np.argsort(((base[cand] - q) ** 2).sum(1))[:k_nn]]
    return top, len(cand)

exact = [np.argsort(((base - q) ** 2).sum(1))[:k_nn] for q in queries]         # brute force ground truth
rows = []
for nprobe in [1, 2, 4, 8, 16, 64]:
    rec, scanned = [], []
    for q, truth in zip(queries, exact):
        found, n_scanned = ivf_search(q, nprobe)
        rec.append(len(set(found) & set(truth)) / k_nn); scanned.append(n_scanned / len(base))
    rows.append((nprobe, np.mean(rec), np.mean(scanned)))
    print(f"nprobe={nprobe:>2}: recall@10 {np.mean(rec):.3f}, vectors scanned {np.mean(scanned):6.1%}")

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot([r[2] for r in rows], [r[1] for r in rows], "o-", lw=2)
for nprobe, rc, sc in rows:
    ax.annotate(f"nprobe={nprobe}", (sc, rc), textcoords="offset points", xytext=(6, -12), fontsize=8)
ax.set(xscale="log", title="IVF: recall vs work (20k vectors, 64 k-means lists)", xlabel="share of vectors scanned (log)", ylabel="recall@10")
plt.show()
recalls = [r[1] for r in rows]
assert all(a <= b + 1e-9 for a, b in zip(recalls, recalls[1:])) and np.isclose(recalls[-1], 1.0) and rows[2][2] < 0.2

A few percent of the vectors scanned already gives high recall, and probing every list is exact brute force again. Compared with **HNSW** (a navigable graph): IVF is lighter on memory, faster to build and easy to combine with compression; HNSW usually gives better recall at the same latency but costs more memory and build time. **Product quantisation** uses k-means again: split each vector into sub-vectors and replace each with the id of its nearest centroid in a small per-subspace codebook ([vector search notebook](../09_rag/03_vector_search_faiss_hnsw.ipynb)).

## Try it yourself

**1.** PCA as compression: reconstruct digit images from the first 5, 15 and 40 components and plot them next to the original. Check that the reconstruction error falls as you keep more components.

In [ ]:
# Solution — try it yourself first, then run this
fig, axes = plt.subplots(1, 4, figsize=(7, 2))
errors = []
for ax, k in zip(axes, [5, 15, 40, 64]):
    recon = Xd.mean(axis=0) + (Xc @ Vt[:k].T) @ Vt[:k]            # project down, then back up
    errors.append(np.mean((recon - Xd) ** 2))
    ax.imshow(recon[0].reshape(8, 8), cmap="Greys"); ax.set(title=f"{k} comps" if k < 64 else "all 64", xticks=[], yticks=[])
plt.show()
print("mean squared reconstruction error:", np.round(errors, 3))
assert errors[0] > errors[1] > errors[2] and errors[-1] < 1e-8

**2.** DBSCAN's `eps` is its most sensitive knob. Sweep it on the moons data and print the number of clusters and noise points for each value. What happens when eps is too small, and too large?

In [ ]:
# Solution — try it yourself first, then run this
for eps in [0.03, 0.08, 0.15, 0.3, 0.6]:
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xmo)
    n_clusters = len(set(lab)) - (1 if -1 in lab else 0)
    print(f"eps={eps:<4}: {n_clusters:2d} clusters, {np.sum(lab == -1):3d} noise points, ARI {adjusted_rand_score(ymo, lab):.2f}")
# too small: everything is noise or shattered into many tiny clusters; too large: both moons merge into one

**3.** Scaling before clustering. Stretch the second feature of the blob data by 20× and run k-means with k=4 with and without `StandardScaler`. Compare the adjusted Rand index against the true blobs.

In [ ]:
# Solution — try it yourself first, then run this
X_stretched = Xb * np.array([1.0, 20.0])
raw = KMeans(4, n_init=10, random_state=0).fit_predict(X_stretched)
scaled = KMeans(4, n_init=10, random_state=0).fit_predict(StandardScaler().fit_transform(X_stretched))
print(f"ARI unscaled {adjusted_rand_score(yb, raw):.2f}   ARI standardised {adjusted_rand_score(yb, scaled):.2f}")

## Say it in an interview

- **k-means in 30 seconds:** "Choose k centres, assign each point to the nearest, move each centre to its points' mean, repeat. It minimises within-cluster squared distance, always converges, but only to a local optimum, so use k-means++ starts and several restarts. It assumes round, similar-sized clusters and needs k: pick it with the elbow or silhouette, then sanity-check by reading clusters."
- **DBSCAN:** density-based, finds arbitrary shapes and labels outliers, needs `eps` and `min_samples` instead of k; struggles when densities vary (HDBSCAN fixes that).
- **PCA in 30 seconds:** "Centre the data and take the SVD; the right singular vectors are the directions of maximum variance, and S²/(n−1) gives the variance along each. Keep enough components for, say, 95% of the variance. It is linear, fast, has a `transform`, and is sign-ambiguous. Standardise first if features have different units."
- **t-SNE / UMAP:** for pictures only. Cluster sizes and inter-cluster distances are not meaningful.
- **Vector-search link:** IVF = k-means centroids plus inverted lists, searched with `nprobe` lists (recall vs latency); product quantisation = k-means codebooks per sub-vector.
- **Traps:** clustering unscaled features; trusting k-means on elongated or nested shapes; evaluating clusters only by inertia (it always improves with k); reading t-SNE distances; comparing PCA component signs.

## Next

- [09 · Feature engineering and pipelines](09_feature_engineering_and_pipelines.ipynb): scaling, encoding and leakage-safe pipelines.
- [Embeddings and similarity](../07_genai_foundations/02_embeddings_and_similarity.ipynb), [vector search: FAISS and HNSW](../09_rag/03_vector_search_faiss_hnsw.ipynb), [chunking and semantic chunking](../09_rag/01_chunking_strategies_and_semantic_chunking.ipynb).
- Theory: [ml_fundamentals course](../../ml_fundamentals/index.html) (chapter 10), [interview bank](../../ai_interview_prep/index.html).